# Emotion-Aware Response Generation
**Pipeline:** User message → RoBERTa (6-class emotion) → Mistral-7B-Instruct-v0.2 4-bit (empathetic reply)

- No HuggingFace token required
- Runs on Colab T4 GPU (~10GB VRAM total for both models)
- Mistral loaded in 4-bit quantization via `bitsandbytes`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip -q install transformers torch accelerate bitsandbytes pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00


In [ ]:
import torch
import json
import textwrap
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device: cuda
GPU: Tesla T4
VRAM available: 15.6 GB


## 1. Load RoBERTa Emotion Classifier
Same model you trained — loaded from your Drive.

In [ ]:
# ── Paths — adjust DRIVE_PROJECT_ROOT if needed ───────────────────────────────
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Emotional Aware Chatbot')
EXP_ROOT           = DRIVE_PROJECT_ROOT / 'experiments' / 'goemotions_roberta_full_pipeline'
PRIMARY_MODEL_DIR  = EXP_ROOT / 'models' / 'best_model'
REPORTS_DIR        = EXP_ROOT / 'reports'
RESPONSE_DIR       = REPORTS_DIR / 'response_generation'
RESPONSE_DIR.mkdir(parents=True, exist_ok=True)

# Fallback to latest checkpoint if best_model doesn't exist
import re
def has_model_files(p):
    return p.exists() and (p/'config.json').exists() and (
        (p/'model.safetensors').exists() or (p/'pytorch_model.bin').exists())

def find_latest_checkpoint(base):
    if not base.exists(): return None
    ckpts = [p for p in base.glob('checkpoint-*') if p.is_dir() and has_model_files(p)]
    return max(ckpts, key=lambda p: int(re.search(r'checkpoint-(\d+)', p.name).group(1))) if ckpts else None

if has_model_files(PRIMARY_MODEL_DIR):
    ROBERTA_DIR = PRIMARY_MODEL_DIR
else:
    ckpt = find_latest_checkpoint(EXP_ROOT / 'results' / 'trainer_runs') or \
           find_latest_checkpoint(DRIVE_PROJECT_ROOT / 'results' / 'roberta_runs')
    if ckpt is None:
        raise FileNotFoundError('No trained RoBERTa model found. Run training notebook first.')
    ROBERTA_DIR = ckpt

print('Loading RoBERTa from:', ROBERTA_DIR)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_DIR)
roberta_model     = AutoModelForSequenceClassification.from_pretrained(ROBERTA_DIR)
roberta_model.to(device).eval()

id2label = roberta_model.config.id2label
LABELS   = [id2label[i] for i in range(len(id2label))]
print('Emotion labels:', LABELS)
print('RoBERTa loaded ✅')


Loading RoBERTa from: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/models/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Emotion labels: ['Anger', 'Fear', 'Joy', 'Love', 'Neutral', 'Sadness']
RoBERTa loaded ✅


## 2. Load Mistral-7B-Instruct-v0.2 in 4-bit
Uses `TheBloke/Mistral-7B-Instruct-v0.2-GGUF` style quantization via `bitsandbytes`.
No HuggingFace token required. Downloads ~4.5GB on first run, cached after that.

In [ ]:
MISTRAL_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading Mistral-7B-Instruct-v0.2 in 4-bit... (first run downloads ~4.5GB)')
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_ID)
mistral_model     = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
mistral_model.eval()

mistral_pipe = pipeline(
    'text-generation',
    model=mistral_model,
    tokenizer=mistral_tokenizer,
    return_full_text=False,   # only return generated text, not the prompt
)

print('Mistral-7B loaded ✅')
if device == 'cuda':
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used: {used:.1f} / {total:.1f} GB')


Loading Mistral-7B-Instruct-v0.2 in 4-bit... (first run downloads ~4.5GB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral-7B loaded ✅
VRAM used: 4.6 / 15.6 GB


## 3. Core Pipeline Functions
- `detect_emotion()` — RoBERTa inference, returns label + confidence + all scores
- `build_prompt()` — constructs Mistral instruction prompt conditioned on emotion
- `generate_response()` — full pipeline: text in, empathetic reply out

In [ ]:
def detect_emotion(text: str, max_length: int = 128) -> dict:
    """
    Run RoBERTa on a single text.
    Returns: {'label': str, 'confidence': float, 'scores': {label: prob}}
    """
    enc = roberta_tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        logits = roberta_model(**enc).logits
        probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

    pred_idx = int(probs.argmax())
    return {
        'label'     : LABELS[pred_idx],
        'confidence': float(probs[pred_idx]),
        'scores'    : {lbl: float(p) for lbl, p in zip(LABELS, probs)},
    }


In [ ]:
# Emotion-specific system instructions — tune these to adjust tone
EMOTION_SYSTEM_PROMPTS = {
    'Anger'  : (
        'You are a calm, empathetic assistant. The user is feeling angry or frustrated. '
        'Acknowledge their frustration without judgment, help them feel heard, and gently '
        'offer a constructive or calming perspective. Do not be dismissive.'
    ),
    'Fear'   : (
        'You are a reassuring and supportive assistant. The user is feeling anxious or afraid. '
        'Validate their concern, offer grounding and reassurance, and help them feel safe. '
        'Keep your tone warm and steady.'
    ),
    'Joy'    : (
        'You are an enthusiastic and warm assistant. The user is feeling happy or excited. '
        'Match their positive energy, celebrate with them, and keep the conversation uplifting.'
    ),
    'Love'   : (
        'You are a warm and caring assistant. The user is expressing love or affection. '
        'Respond with genuine warmth, kindness, and positivity.'
    ),
    'Neutral': (
        'You are a helpful, friendly assistant. The user seems calm and neutral. '
        'Respond clearly and helpfully in a conversational tone.'
    ),
    'Sadness': (
        'You are a compassionate and gentle assistant. The user is feeling sad or down. '
        'Acknowledge their pain with empathy, let them know they are not alone, and offer '
        'gentle support without trying to immediately fix everything.'
    ),
}

DEFAULT_SYSTEM = 'You are a helpful and empathetic assistant.'


def build_prompt(user_text: str, emotion: str, confidence: float) -> str:
    """
    Build a Mistral [INST] prompt conditioned on the detected emotion.
    Mistral-Instruct format: <s>[INST] <<SYS>>...system...<</SYS>> user_message [/INST]
    """
    system = EMOTION_SYSTEM_PROMPTS.get(emotion, DEFAULT_SYSTEM)

    # Inject detected emotion context into the user turn
    emotion_context = (
        f'[Detected emotion: {emotion} (confidence: {confidence:.0%})]\n'
        f'User message: {user_text}'
    )

    prompt = (
        f'<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n'
        f'{emotion_context} [/INST]'
    )
    return prompt


In [ ]:
def generate_response(
    user_text      : str,
    max_new_tokens : int   = 256,
    temperature    : float = 0.7,
    top_p          : float = 0.9,
    repetition_penalty: float = 1.1,
    verbose        : bool  = True,
) -> dict:
    """
    Full pipeline: text → emotion → empathetic response.
    Returns a dict with emotion info + generated reply.
    """
    # Step 1: Detect emotion
    emotion_result = detect_emotion(user_text)
    emotion    = emotion_result['label']
    confidence = emotion_result['confidence']

    # Step 2: Build prompt
    prompt = build_prompt(user_text, emotion, confidence)

    # Step 3: Generate with Mistral
    output = mistral_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=mistral_tokenizer.eos_token_id,
    )

    reply = output[0]['generated_text'].strip()

    result = {
        'user_text'   : user_text,
        'emotion'     : emotion,
        'confidence'  : confidence,
        'all_scores'  : emotion_result['scores'],
        'reply'       : reply,
    }

    if verbose:
        print('─' * 60)
        print(f'👤 User     : {user_text}')
        print(f'🧠 Emotion  : {emotion} ({confidence:.1%} confidence)')
        print(f'📊 All scores: ' + ', '.join(f'{k}: {v:.2f}' for k, v in emotion_result['scores'].items()))
        print(f'🤖 Response :\n')
        for line in textwrap.wrap(reply, width=70):
            print(f'   {line}')
        print('─' * 60)

    return result


## 4. Test with Sample Inputs
One example per emotion class to verify the pipeline works end-to-end.

In [ ]:
TEST_INPUTS = [
    'I just got promoted at work today! I cannot believe it!',          # Joy
    'I am so tired of being ignored. Nobody listens to me.',            # Anger
    'I miss my grandmother so much. She passed away last month.',       # Sadness
    'I am terrified about my surgery tomorrow. What if something goes wrong?',  # Fear
    'I love spending time with my family. They mean everything to me.', # Love
    'Can you tell me what the weather is like today?',                  # Neutral
]

results = []
for text in TEST_INPUTS:
    r = generate_response(text)
    results.append(r)
    print()  # spacing


Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'max_new_tokens', 'top_p', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : I just got promoted at work today! I cannot believe it!
🧠 Emotion  : Joy (99.6% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 1.00, Love: 0.00, Neutral: 0.00, Sadness: 0.00
🤖 Response :

   Congratulations on your promotion! That's absolutely fantastic news.
   I'm thrilled for you and your well-deserved success. Your hard work
   and dedication have truly paid off. Keep up the great work and
   continue to reach for the stars! Is there anything specific you're
   looking forward to in your new role?
────────────────────────────────────────────────────────────



Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : I am so tired of being ignored. Nobody listens to me.
🧠 Emotion  : Sadness (99.9% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 0.00, Love: 0.00, Neutral: 0.00, Sadness: 1.00
🤖 Response :

   I'm really sorry to hear that you're feeling ignored and unheard right
   now. It can be a very painful and isolating experience. Please know
   that you're not alone in this feeling. Many people go through periods
   of feeling overlooked and unappreciated. It's important to remember
   that everyone has the capacity for error and sometimes we may not
   communicate effectively or give our full attention to those around us.
   However, it's also crucial that we acknowledge and address these
   feelings when they arise. If you feel comfortable, perhaps you could
   try expressing your thoughts and emotions to someone you trust? They
   might be able to provide you with the validation and understanding
   that you'r

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : I miss my grandmother so much. She passed away last month.
🧠 Emotion  : Sadness (100.0% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 0.00, Love: 0.00, Neutral: 0.00, Sadness: 1.00
🤖 Response :

   I'm really sorry for your loss. I can only imagine how difficult it is
   for you right now. Losing someone we love deeply can be one of the
   most painful experiences. It's normal to feel sadness and longing for
   their presence. Please know that you are not alone in this. Many
   people have felt the same way after losing a loved one. Take your time
   to grieve and allow yourself to feel your emotions fully. Remember the
   happy memories and cherish them. If you ever need someone to talk to
   or just want to share your thoughts, I'm here for you. You're not
   alone in this journey.
────────────────────────────────────────────────────────────



Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : I am terrified about my surgery tomorrow. What if something goes wrong?
🧠 Emotion  : Fear (99.9% confidence)
📊 All scores: Anger: 0.00, Fear: 1.00, Joy: 0.00, Love: 0.00, Neutral: 0.00, Sadness: 0.00
🤖 Response :

   I'm really sorry to hear that you're feeling scared about your
   upcoming surgery. It's completely normal to have fears and concerns
   before a medical procedure. I want to assure you that the medical team
   has extensive training and experience in performing surgeries. They
   will do everything in their power to ensure your safety and comfort
   throughout the process.  I understand that it's hard to let go of
   these worries, but try to focus on the fact that thousands of people
   undergo similar procedures every day with positive outcomes. You've
   made it this far, and I believe in your strength and resilience.  If
   it helps, you can prepare yourself by learning as much as possible
   ab

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : I love spending time with my family. They mean everything to me.
🧠 Emotion  : Joy (99.6% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 1.00, Love: 0.00, Neutral: 0.00, Sadness: 0.00
🤖 Response :

   That's wonderful to hear! Family is truly a source of joy and
   happiness. I'm so glad that you cherish the moments spent with them.
   They bring us warmth, support, and create beautiful memories. Is there
   a specific activity or tradition that brings you closer as a family?
   I'd be delighted to know more about it!
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
👤 User     : Can you tell me what the weather is like today?
🧠 Emotion  : Neutral (99.6% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 0.00, Love: 0.00, Neutral: 1.00, Sadness: 0.00
🤖 Response :

   Of course! I'd be happy to help you with that. However, I need

## 5. Batch Processing on Test Set
Run the full pipeline on a sample of your test CSV and save results.

In [ ]:
TEST_PATH  = EXP_ROOT / 'data' / 'processed' / 'test.csv'
BATCH_N    = 50   # number of test samples to run (increase if you have time)

test_df = pd.read_csv(TEST_PATH)
sample  = test_df.sample(n=min(BATCH_N, len(test_df)), random_state=42).reset_index(drop=True)

print(f'Running pipeline on {len(sample)} test samples...')
batch_results = []
for i, row in sample.iterrows():
    r = generate_response(row['text'], verbose=False)
    r['true_label'] = row.get('final_emotion', 'NA')
    r['emotion_correct'] = (r['emotion'] == r['true_label'])
    batch_results.append(r)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(sample)} done...')

# Flatten all_scores into separate columns
results_df = pd.DataFrame(batch_results)
scores_df  = pd.json_normalize(results_df['all_scores'])
results_df = pd.concat([results_df.drop(columns='all_scores'), scores_df], axis=1)

out_path = RESPONSE_DIR / 'batch_responses.csv'
results_df.to_csv(out_path, index=False)

accuracy = results_df['emotion_correct'].mean()
print(f'\nBatch complete. Emotion detection accuracy on sample: {accuracy:.1%}')
print(f'Saved to: {out_path}')
results_df[['user_text','true_label','emotion','confidence','reply']].head(5)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running pipeline on 50 test samples...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati

  10/50 done...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

...

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  40/50 done...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  50/50 done...

Batch complete. Emotion detection accuracy on sample: 90.0%
Saved to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/response_generation/batch_responses.csv


,user_text,true_label,emotion,confidence,reply
0,Do something you’ll love and you’ll never work...,Anger,Anger,0.986386,I'm really sorry that the quote you came acros...
1,> the heartbreak of a nice older truck being r...,Anger,Anger,0.998475,I'm really sorry to hear that you're feeling u...
2,"Sorry, no trivia today. Calendar had a fun fac...",Neutral,Joy,0.974713,"That's totally fine, user! It's great that you..."
3,Damn you! Now all this dirty money is flooding...,Anger,Anger,0.986789,I'm really sorry to hear that you're feeling f...
4,I’m to going die right now I’m so scared tears...,Fear,Fear,0.999946,I'm really sorry that you're feeling this way ...


## 6. Interactive Single-Turn Chat
Type any message and get an emotion-aware response.

In [ ]:
# ── Run this cell repeatedly to chat ─────────────────────────────────────────
user_input = input('You: ')
if user_input.strip():
    generate_response(user_input)
else:
    print('Please type something!')


You: India won the match, I am going to celebrate 


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────────────────
👤 User     : India won the match, I am going to celebrate 
🧠 Emotion  : Joy (99.9% confidence)
📊 All scores: Anger: 0.00, Fear: 0.00, Joy: 1.00, Love: 0.00, Neutral: 0.00, Sadness: 0.00
🤖 Response :

   Congratulations on India's victory! That's wonderful news. I'm
   thrilled for you and your team. Let's celebrate together! What's your
   favorite way to celebrate a win? I'd love to hear about it. Maybe we
   can even share some dance moves or songs that get you in the mood. Go
   India!
────────────────────────────────────────────────────────────


## 7. Multi-Turn Conversation (Memory)
Keeps conversation history so Mistral can refer back to previous messages.
Emotion is re-detected on every new user turn.

In [ ]:
class EmotionChatSession:
    """
    Multi-turn conversation with per-turn emotion detection.
    Maintains full history in Mistral's [INST] format.
    """

    def __init__(self, max_history_turns: int = 5):
        self.history           = []   # list of (user_text, emotion, reply)
        self.max_history_turns = max_history_turns

    def _build_multiturn_prompt(self, user_text: str, emotion: str, confidence: float) -> str:
        system = EMOTION_SYSTEM_PROMPTS.get(emotion, DEFAULT_SYSTEM)
        prompt = f'<s>'

        # Add past turns (trimmed to max_history_turns)
        past = self.history[-self.max_history_turns:]
        for turn in past:
            prompt += (
                f'[INST] [Detected emotion: {turn["emotion"]}] {turn["user_text"]} [/INST] '
                f'{turn["reply"]} </s>'
            )

        # Add current turn
        emotion_ctx = f'[Detected emotion: {emotion} ({confidence:.0%})]\nUser: {user_text}'
        prompt += f'[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{emotion_ctx} [/INST]'
        return prompt

    def chat(
        self,
        user_text      : str,
        max_new_tokens : int   = 256,
        temperature    : float = 0.7,
        top_p          : float = 0.9,
    ) -> str:
        emotion_result = detect_emotion(user_text)
        emotion    = emotion_result['label']
        confidence = emotion_result['confidence']

        prompt = self._build_multiturn_prompt(user_text, emotion, confidence)

        output = mistral_pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
        reply = output[0]['generated_text'].strip()

        self.history.append({'user_text': user_text, 'emotion': emotion, 'reply': reply})

        print(f'🧠 [{emotion} {confidence:.0%}]')
        print(f'🤖 {reply}\n')
        return reply

    def reset(self):
        self.history = []
        print('Conversation reset.')


# Start a session
session = EmotionChatSession(max_history_turns=5)
print('Session ready. Call session.chat("your message") to talk.')
print('Call session.reset() to start over.\n')


Session ready. Call session.chat("your message") to talk.
Call session.reset() to start over.



In [ ]:
session.chat('I just found out my best friend is moving to another country.')


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 [Love 94%]
🤖 Oh no, I'm so sorry to hear that your best friend will be moving away. It must be difficult for you. Remember though, even if they're physically far away, the bond of friendship can remain strong through regular communication and shared memories. Wishing you both all the best in this new chapter of life. Stay positive and know that distance makes the heart grow fonder!



"Oh no, I'm so sorry to hear that your best friend will be moving away. It must be difficult for you. Remember though, even if they're physically far away, the bond of friendship can remain strong through regular communication and shared memories. Wishing you both all the best in this new chapter of life. Stay positive and know that distance makes the heart grow fonder!"

In [ ]:
session.chat('Yeah, we have been friends for 15 years. I feel so lost.')


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 [Neutral 64%]
🤖 I completely understand how you might be feeling right now, given the length and depth of your friendship with your best friend. It's natural to feel a sense of loss when someone we care about moves away. Try not to focus too much on the physical separation, instead cherish the memories and moments you've shared together. Keeping in touch regularly through calls, messages or social media can help keep the connection alive. It may take some time to adjust, but remember that change can also bring new opportunities and experiences. Do you want to talk more about it or perhaps look up ways to maintain long-distance friendships?



"I completely understand how you might be feeling right now, given the length and depth of your friendship with your best friend. It's natural to feel a sense of loss when someone we care about moves away. Try not to focus too much on the physical separation, instead cherish the memories and moments you've shared together. Keeping in touch regularly through calls, messages or social media can help keep the connection alive. It may take some time to adjust, but remember that change can also bring new opportunities and experiences. Do you want to talk more about it or perhaps look up ways to maintain long-distance friendships?"

In [ ]:
session.chat('You are right. Maybe I should plan a visit soon.')


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 [Love 52%]
🤖 Absolutely! Planning a visit is an excellent idea to make the most of your time together before your friend moves. It could be a special trip filled with adventures and creating new memories. The anticipation of the visit can also provide something exciting to look forward to during this transition. If a visit isn't possible at the moment, consider sending thoughtful gifts or letters as a way to show your support and love from afar. Remember, the strength of your friendship remains unchanged despite the geographical distance. Good luck with your plans and keep me posted! 😊



"Absolutely! Planning a visit is an excellent idea to make the most of your time together before your friend moves. It could be a special trip filled with adventures and creating new memories. The anticipation of the visit can also provide something exciting to look forward to during this transition. If a visit isn't possible at the moment, consider sending thoughtful gifts or letters as a way to show your support and love from afar. Remember, the strength of your friendship remains unchanged despite the geographical distance. Good luck with your plans and keep me posted! 😊"

## 8. Save Pipeline Config
Saves a JSON config that the future Gradio/Streamlit chatbot app can load.

In [ ]:
config = {
    'roberta_model_path' : str(ROBERTA_DIR),
    'mistral_model_id'   : MISTRAL_MODEL_ID,
    'labels'             : LABELS,
    'quantization'       : '4-bit NF4',
    'max_new_tokens'     : 256,
    'temperature'        : 0.7,
    'top_p'              : 0.9,
    'repetition_penalty' : 1.1,
    'max_history_turns'  : 5,
    'emotion_prompts'    : EMOTION_SYSTEM_PROMPTS,
}

config_path = RESPONSE_DIR / 'pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Pipeline config saved to:', config_path)
print('\nNext step: Build Gradio chatbot UI using this config.')


✅ Pipeline config saved to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/reports/response_generation/pipeline_config.json

Next step: Build Gradio chatbot UI using this config.
